# Week 33, Day 4: Generics, TypeVar and Callable

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w033d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def first[T](items: list[T]) -> T:
    return items[0]

print(first([3, 1]), first(["Ada", "Bo"]))
print(first.__type_params__, first.__annotations__["return"])

In [ ]:
def invert[K, V](d: dict[K, V]) -> dict[V, K]:
    return {v: k for k, v in d.items()}

def pair_up[A, B](xs: list[A], ys: list[B]) -> list[tuple[A, B]]:
    return list(zip(xs, ys))

print(invert({"Ada": 1, "Bo": 2}))
print(pair_up(["Lagos", "Leeds"], [15.4, 0.8]))

In [ ]:
from collections.abc import Iterable, Sized

def longest[S: Sized](items: Iterable[S]) -> S | None:
    best = None
    for item in items:
        if best is None or len(item) > len(best):
            best = item
    return best

print(longest(["fig", "plantain", "yam"]), longest([[1], [1, 2, 3], []]), longest([]))

In [ ]:
def biggest[N: (int, float)](a: N, b: N) -> N:
    return a if a > b else b

print(biggest(2, 3), biggest(2.5, 1.0))

In [ ]:
class Stack[T]:
    def __init__(self) -> None:
        self._items: list[T] = []

    def push(self, item: T) -> None:
        self._items.append(item)

    def pop(self) -> T:
        if not self._items:
            raise IndexError("pop from an empty stack")
        return self._items.pop()

    def __len__(self) -> int:
        return len(self._items)

plates = Stack[int]()
plates.push(3)
plates.push(5)
print(plates.pop() + 1, len(plates))
names: Stack[str] = Stack()
names.push("Ada")
print(names.pop().upper())

In [ ]:
from collections.abc import Callable, Iterable

def apply_twice(f: Callable[[int], int], x: int) -> int:
    return f(f(x))

def apply_all[T, R](f: Callable[[T], R], items: Iterable[T]) -> list[R]:
    return [f(item) for item in items]

print(apply_twice(lambda n: n * 3, 2))
print(apply_all(len, ["fig", "plantain"]), apply_all(str.upper, ["ada", "bo"]))

> **Key idea.** `Any` loses the link between a function's input and output types; a type variable keeps it: `def first[T](items: Sequence[T]) -> T` (older code: `T = TypeVar("T")`). Type checkers work out `T` for each call. Bound a type variable (`[S: Sized]`) to use its methods, or constrain it to a list of types (`[N: (int, float)]`). Generic classes (`class Stack[T]:`, older `Generic[T]`) are used as `Stack[int]`, and the parameters make no difference when the program runs. `Callable[[A, B], R]` describes functions passed as arguments; combine it with type variables for general helpers.

### Exercise: Generic functions

Write two generic functions with the `def f[T]` syntax:

- `last[T](items: Sequence[T]) -> T`, the last item, raising `IndexError` for an empty sequence;
- `pair_up[A, B](xs: list[A], ys: list[B]) -> list[tuple[A, B]]`, pairing items in order; lists of different lengths raise `ValueError`.

Import `Sequence` from `collections.abc`.

In [ ]:
from collections.abc import Sequence

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_generic():\n    \"generic\"\n    from typing import get_type_hints as h\n    from collections.abc import Sequence as Seq\n    assert len(last.__type_params__) == 1 and len(pair_up.__type_params__) == 2, \"use def last[T] and def pair_up[A, B]\"\n    T, = last.__type_params__\n    A, B = pair_up.__type_params__\n    assert h(last) == {\"items\": Seq[T], \"return\": T}, f\"last: {h(last)}\"\n    assert h(pair_up) == {\"xs\": list[A], \"ys\": list[B], \"return\": list[tuple[A, B]]}, f\"pair_up: {h(pair_up)}\"\n    assert last([1, 2, 3]) == 3 and last(\"abc\") == \"c\" and last((None,)) is None\n    assert pair_up([\"a\", \"b\"], [1, 2]) == [(\"a\", 1), (\"b\", 2)] and pair_up([], []) == []\n    for f in [lambda: last([]), lambda: pair_up([1], [])]:\n        try:\n            f()\n            assert False, \"should raise\"\n        except (IndexError, ValueError):\n            pass")

### Exercise: A generic queue

Write a generic class `Queue[T]` (first in, first out) with fully annotated methods: `join(item: T) -> None`, `serve() -> T` (raising `IndexError` if empty), `peek() -> T | None` (the next item without removing it, or `None`), `__len__() -> int`, and `__init__(self) -> None`. Keep the items in a `collections.deque` (Week 20).

In [ ]:
from collections import deque

class Queue[T]:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_queue():\n    \"a queue\"\n    from typing import get_type_hints as h\n    Q = ns[\"Queue\"]\n    T, = Q.__type_params__\n    assert h(Q.join) == {\"item\": T, \"return\": type(None)} and h(Q.serve) == {\"return\": T} and h(Q.peek) == {\"return\": T | None}\n    assert h(Q.__len__) == {\"return\": int} and h(Q.__init__) == {\"return\": type(None)}\n    q = Q[str]()\n    q.join(\"Ada\")\n    q.join(\"Bo\")\n    assert q.peek() == \"Ada\" and len(q) == 2 and q.serve() == \"Ada\" and q.serve() == \"Bo\" and q.peek() is None and len(q) == 0\n    try:\n        q.serve()\n        assert False, \"serving an empty queue should raise IndexError\"\n    except IndexError:\n        pass")

### Exercise: Functions in, functions out

Write these generic functions, with hints using `Callable` and `Iterable` from `collections.abc`:

- `apply_all[T, R](f: Callable[[T], R], items: Iterable[T]) -> list[R]`;
- `keep[T](test: Callable[[T], bool], items: Iterable[T]) -> list[T]`, the items for which `test` is True;
- `compose[A, B, C](f: Callable[[A], B], g: Callable[[B], C]) -> Callable[[A], C]`, which returns a **new function** that applies `f` and then `g`.

In [ ]:
from collections.abc import Callable, Iterable

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_callables():\n    \"callables\"\n    from typing import get_type_hints as h\n    from collections.abc import Callable as C, Iterable as I\n    T, R = apply_all.__type_params__\n    assert h(apply_all) == {\"f\": C[[T], R], \"items\": I[T], \"return\": list[R]}, f\"apply_all: {h(apply_all)}\"\n    K, = keep.__type_params__\n    assert h(keep) == {\"test\": C[[K], bool], \"items\": I[K], \"return\": list[K]}\n    A, B, D = compose.__type_params__\n    assert h(compose) == {\"f\": C[[A], B], \"g\": C[[B], D], \"return\": C[[A], D]}\n    assert apply_all(len, [\"fig\", \"yam\"]) == [3, 3] and keep(str.isdigit, [\"1\", \"a\", \"22\"]) == [\"1\", \"22\"]\n    shout_len = compose(str.upper, len)\n    assert shout_len(\"ada\") == 3 and compose(lambda n: n + 1, str)(41) == \"42\" and keep(bool, range(3)) == [1, 2]")

### Exercise: Bounded and constrained

Write:

- `longest[S: Sized](items: Iterable[S]) -> S | None`: the first of the longest items, or `None` if there are none;
- `clamp[N: (int, float)](value: N, low: N, high: N) -> N`: `value` kept between `low` and `high` (raise `ValueError` if `low > high`).

Import `Iterable` and `Sized` from `collections.abc`.

In [ ]:
from collections.abc import Iterable, Sized

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_bounded():\n    \"bounds\"\n    from collections.abc import Sized as Sz\n    S, = longest.__type_params__\n    N, = clamp.__type_params__\n    assert S.__bound__ is Sz, \"S should be bound to Sized\"\n    assert N.__constraints__ == (int, float), \"N should be constrained to (int, float)\"\n    assert longest([\"fig\", \"yam\", \"plantain\", \"okra\"]) == \"plantain\" and longest([\"ab\", \"cd\"]) == \"ab\" and longest([]) is None\n    assert longest([{1}, {1, 2}]) == {1, 2} and clamp(15, 0, 10) == 10 and clamp(-2.5, -1.0, 1.0) == -1.0 and clamp(5, 0, 10) == 5\n    try:\n        clamp(1, 5, 0)\n        assert False, \"low > high should raise ValueError\"\n    except ValueError:\n        pass")

### Bug hunt: The wrong kind of function

mypy reports `Argument 1 to "apply_twice" has incompatible type "Callable[[str], str]"; expected "Callable[[int], int]"  [arg-type]`, and the program crashes with a `TypeError`. The aim is to triple 2 twice, printing `18`. Fix the call.

In [ ]:
from collections.abc import Callable

def apply_twice(f: Callable[[int], int], x: int) -> int:
    return f(f(x))

print(apply_twice(str.upper, 2))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"tripled twice\"\n    expect_output(\"18\")")

### Bug hunt: A string in a stack of ints

mypy reports `Argument 1 to "push" of "Stack" has incompatible type "str"; expected "int"  [arg-type]`, and adding up the stack crashes with `TypeError`. Fix the line mypy points to, so it prints `10`.

In [ ]:
class Stack[T]:
    def __init__(self) -> None:
        self._items: list[T] = []

    def push(self, item: T) -> None:
        self._items.append(item)

    def total_of(self) -> list[T]:
        return list(self._items)

boxes = Stack[int]()
boxes.push(3)
boxes.push("three")
boxes.push(4)
print(sum(boxes.total_of()))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the total\"\n    expect_output(\"10\")")

### Bug hunt: A key that isn't a name

mypy reports `Incompatible return value type (got "Player", expected "str")  [return-value]` for `by_name`, and sorting crashes with `TypeError: '<' not supported between instances of 'Player' and 'Player'`. Fix `by_name` so the program prints `['Ade', 'Kane', 'Osimhen']`.

In [ ]:
from collections.abc import Callable
from dataclasses import dataclass

@dataclass
class Player:
    name: str
    goals: int

def by_name(p: Player) -> str:
    return p

def ranked(players: list[Player], key: Callable[[Player], str]) -> list[str]:
    return [p.name for p in sorted(players, key=key)]

squad = [Player("Osimhen", 20), Player("Ade", 12), Player("Kane", 20)]
print(ranked(squad, by_name))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"sorted\"\n    expect_output(\"['Ade', 'Kane', 'Osimhen']\")")

### Puzzle: A generic lookup

Arrange the lines into a generic function that returns a dictionary's value for a key, or a default of the same type. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
from collections.abc import Callable

def twice[T](f: Callable[[T], T], x: T) -> T:
    return f(f(x))

class Box[T]:
    def __init__(self, item: T) -> None:
        self.item = item

p = ___   # twice(lambda s: s + "!", "hi")
q = ___   # twice(abs, -4)
r = ___   # Box[int]("not an int").item
s = ___   # len(twice.__type_params__)
t = ___   # type(Box[str]("x")).__name__

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert (p, q, r) == (\"hi!!\", 4, \"not an int\"), \"p to r\"\n    assert (s, t) == (1, \"Box\")")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a generic loading cache

Write a generic class `LoadingCache[K, V]` that fetches values with a loader function and remembers up to `capacity` of them, forgetting the least recently used (like Week 31's Forge bonus, now fully typed).

- `__init__(self, loader: Callable[[K], V], capacity: int) -> None` (capacity at least 1, otherwise `ValueError`).
- `get(self, key: K) -> V`: the remembered value (counted in `hits`, and made most recently used), or, if it isn't remembered, the loader's value (counted in `misses`), which is then remembered, forgetting the least recently used if over capacity.
- `peek(self, key: K) -> V | None`: the remembered value without loading, counting or reordering anything.
- `__len__(self) -> int` and `keys(self) -> list[K]`, from least to most recently used.
- A generic **method** `map_values[R](self, f: Callable[[V], R]) -> dict[K, R]`: a dictionary from each remembered key to `f` of its value, in the same order as `keys()`.

Every method must be fully annotated, with `K` and `V` as the class's type parameters.

In [ ]:
from collections.abc import Callable

class LoadingCache[K, V]:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_hints():\n    \"the hints\"\n    from typing import get_type_hints as h\n    from collections.abc import Callable as C\n    L = ns[\"LoadingCache\"]\n    K, V = L.__type_params__\n    assert h(L.__init__) == {\"loader\": C[[K], V], \"capacity\": int, \"return\": type(None)}, f\"__init__: {h(L.__init__)}\"\n    assert h(L.get) == {\"key\": K, \"return\": V} and h(L.peek) == {\"key\": K, \"return\": V | None}\n    assert h(L.__len__) == {\"return\": int} and h(L.keys) == {\"return\": list[K]}\n    R, = L.map_values.__type_params__\n    assert h(L.map_values) == {\"f\": C[[V], R], \"return\": dict[K, R]}\ndef test_cache():\n    \"the cache\"\n    L = ns[\"LoadingCache\"]\n    calls = []\n    def square(n):\n        calls.append(n)\n        return n * n\n    c = L(square, 2)\n    assert [c.get(2), c.get(3), c.get(2), c.get(4), c.get(3)] == [4, 9, 4, 16, 9]\n    assert calls == [2, 3, 4, 3] and (c.hits, c.misses) == (1, 4) and c.keys() == [4, 3] and len(c) == 2\n    assert c.peek(4) == 16 and c.peek(2) is None and c.keys() == [4, 3] and (c.hits, c.misses) == (1, 4)\n    assert c.map_values(str) == {4: \"16\", 3: \"9\"} and list(c.map_values(lambda v: v > 10)) == [4, 3]\n    try:\n        L(square, 0)\n        assert False, \"capacity 0 should raise ValueError\"\n    except ValueError:\n        pass")

## Warm-up

`first[T]` declares a **type parameter** called `T`, and the hints say: the items are a list of `T`, and the result is a `T`. When the program runs, `T` makes no difference at all (it's just stored, like every hint). To a type checker, it means `first(["Ada", "Bo"])` returns a `str` and `first([3, 1])` returns an `int`, worked out separately for each call. That's what today is about.

## What Any loses

How would you hint `first` with what you know so far? `list[int]` is too narrow; it should work for any list. `Any` (Day 2) accepts every list, but look at what mypy thinks the result is:

```python
from typing import Any

def first_any(items: list[Any]) -> Any:
    return items[0]

reveal_type(first_any(["Ada"]))
```

```text
gen.py:6: note: Revealed type is "Any"
```

The caller put strings in, but gets `Any` out, so the checker stops checking from there on: `first_any(["Ada"]) + 1` passes without complaint and crashes when it runs. `Any` breaks the chain. What `first` needs to say is "the result has the same type as the items, whatever that is".

## Type variables

A **type variable** stands for "some type, the same one everywhere it appears in this signature". Since Python 3.12, you declare one in square brackets after the function's name:

```python
from collections.abc import Sequence

def first[T](items: Sequence[T]) -> T:
    return items[0]

name = first(["Ada", "Bo"])
reveal_type(name)
total = first([1, 2]) + "x"
```

```text
gen.py:7: note: Revealed type is "str"
gen.py:8: error: Unsupported operand types for + ("int" and "str")  [operator]
```

For each call, mypy works out what `T` must be from the arguments (`str` for the first, `int` for the second) and uses it for the result. Code like this is called **generic**: written once, checked for each type it's used with.

A signature can have several type variables, each standing for its own type:

To mypy, `invert({"Ada": 1})` has the type `dict[int, str]`. The names `T`, `K`, `V` are conventions, like `i` for a loop counter; longer names are fine.

Before Python 3.12, type variables were made with `typing.TypeVar` at the top of the module, and you'll meet this form constantly in existing code:

```python
from typing import TypeVar

T = TypeVar("T")

def first(items: Sequence[T]) -> T:
    return items[0]
```

Both forms mean the same to a type checker. This course uses the newer one.

## Bounds and constraints

Sometimes "any type" is too much: the function needs to do something with the values. A **bound** says the type must be a particular type or one of its subclasses (or, for a Protocol, have its methods):

Because `S` is bound to `Sized`, mypy accepts `len(item)`, and the result keeps the caller's own type: a list of strings gives back a `str`, not just "something sized".

A **constraint** lists the exact types allowed, written as a tuple:

With constraints, `T` must be exactly one of the listed types for each call, so `biggest(2, 3)` is an `int`, and `biggest("a", "b")` is an error. Bounds are much more common; reach for constraints only when the types really are a short fixed list.

## Generic classes

A class can have type parameters too. That's how the built-in collections work: `list[int]` is the generic class `list` with its parameter filled in. Your own containers can do the same:

`Stack[int]()` makes a stack that mypy checks as holding ints (at run time it's an ordinary `Stack`). So mypy catches this:

```python
s = Stack[int]()
s.push("three")
```

```text
gen.py:2: error: Argument 1 to "push" of "Stack" has incompatible type "str"; expected "int"  [arg-type]
```

In older code you'll see the same class written `class Stack(Generic[T]):`, with `T` made by `TypeVar` and `Generic` imported from `typing`.

## Callable

Functions are values (Week 7), so they get hints too. `Callable[[int, str], bool]` means "a function that takes an `int` and a `str` and returns a `bool`": the argument types in a list, then the return type. Import it from `collections.abc`.

`apply_all` combines both ideas: whatever `T` the function takes must be what the items are, and the result is a list of whatever `R` it returns. For `apply_all(len, ...)` mypy works out a `list[int]`.

Passing the wrong kind of function is caught like any other wrong argument. Here's mypy's report for `apply_twice(str.upper, 3)`:

```text
gen.py:9: error: Argument 1 to "apply_twice" has incompatible type "Callable[[str], str]"; expected "Callable[[int], int]"  [arg-type]
```

`Callable[..., int]` (with three dots) means "any arguments, returning an `int`", for when the arguments don't matter. For callables with keyword arguments or optional parameters, `Callable` isn't expressive enough; a Protocol with a `__call__` method (Week 32 and Week 31 combined) describes them exactly.

## Practice

## Quiz

## Daily challenge

## Before you go

Find a helper function in your own code that works for many types (one that picks, groups, sorts or wraps things), and give it a type variable instead of `Any` or no hints. Tomorrow: static versus runtime types, what's checked when, and how to check data from outside your program while it runs.